In [ ]:
path_prefix = '../..'

import sys
sys.path.append(path_prefix)

import os
import json
from tqdm import tqdm
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from egh_vlm.utils import Qwen3ModelBundle, get_response_qwen3, save_dataset, get_img_path, get_pred

#### Helper Funcs

In [ ]:
def generate_answers_qwen3(
    dataset_path: str, 
    img_folder_path: str,
    prompt_path: str,
    model_bundle: Qwen3ModelBundle,
    dataset_name: str='phd',
    save_path: str=None,
    save_interval: int=20,
    sample_size: int=None,
    specified_ids: list=None
):
    results = []
    processed_ids = []

    # Load dataset
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    if sample_size is not None and len(dataset) > sample_size:
        dataset = dataset[:sample_size]
    
    # Limit dataset by specified_ids if provided
    if specified_ids is not None:
        dataset = [item for item in dataset if item['id'] in specified_ids]

    # Load processed results
    if save_path is not None and os.path.exists(save_path):
        with open(save_path, 'r', encoding='utf-8') as f:
            results = json.load(f)
    processed_ids = [item['id'] for item in results]

    # Load prompt
    with open(prompt_path, 'r', encoding='utf-8') as f:
        prompt = f.read()
    
    # Generate answers
    batch = []

    for item in tqdm(dataset, desc=f'Processing dataset {dataset_name}'):
        if item['id'] in processed_ids:
            continue
        
        img_path = get_img_path(img_folder_path, item['image_id'], dataset=dataset_name)
        question = prompt.format(question=item['question'])
        messages = [
            { 
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': img_path},
                    {'type': 'text', 'text': question}],
            }
        ]
        answer = get_response_qwen3(messages, model_bundle, 64)
        
        # Post-processing
        item['answer'] = answer
        # Assign hallucinated label based on yes/no in the response
        pred = get_pred(answer)
        if pred == 0.5:
            print(f'Item ID {item["id"]} with response: "{answer}" can\'t be classified. Defaulting to non-hallucinated (0).')
            pred = 0
        hallucinated_label = 0 if get_pred(answer) == item['question_gt'] else 1
        item['hallucinated_label'] = hallucinated_label
        batch.append(item)

        # Save intermediate results
        if save_path is not None and len(batch) > save_interval:
            results.extend(batch)
            save_dataset(results, save_path)
            batch.clear()
            
    # Final save
    if save_path is not None and batch:
        results.extend(batch)
        save_dataset(results, save_path)
    print(f'Completed. Dataset size: {len(results)}')
    return results

#### Generate Answers

Qwen/Qwen3-VL-2B-Instruct

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype='auto',
    device_map=device
)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024*1024  
)

model_bundle_qwen3 = Qwen3ModelBundle(model=model, processor=processor, device=device)

In [ ]:
res_qwen3 = generate_answers_qwen3(
    dataset_path=f'{path_prefix}/data/phd/phd_base.json',
    img_folder_path=f'{path_prefix}/data/phd/images',
    prompt_path=f'{path_prefix}/data/phd/prompts/generate.txt',
    model_bundle=model_bundle_qwen3,
    save_path=f'{path_prefix}/data/phd/phd_base_qwen3_vl_2b_instruct.json',
    sample_size=10,
)